In [1]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.5
numpy: 2.5.2


In [2]:
water_systems = pd.read_csv("../data/water_systems_master.csv")

water_systems.head()

,water_system_id,water_system,source_type,source_name,average_daily_water_use_mgd,water_use_year,water_use_type,water_use_status,water_use_source_id,treatment_capacity_mgd,...,capacity_status,capacity_source_id,served_value,served_unit,served_qualifier,served_year,served_source_id,system_scope,research_status,notes
0,CLE_WATER,Cleveland Division of Water / Cleveland Water,great_lake,"Lake Erie, Central Basin",236.00,NaN,derived,available,SRC_CLE_WATER_TREATMENT,540.0,...,available_source_pending,SRC_CLE_CAP_USER,1400000.0,people,greater_than,NaN,SRC_CLE_WATER_CURRENT,interconnected_water_system,core_fields_complete_source_followup,Average daily water use is derived by summing ...
1,AKR_WATER,City of Akron Water Supply,river_reservoir,Upper Cuyahoga River watershed / four-reservoi...,30.44,2023.0,reported,available,SRC_AKR_AIS_2025,67.0,...,available,SRC_AKR_AIS_2025,84950.0,accounts,exact,2023.0,SRC_AKR_AIS_2025,water_system,core_fields_complete,System covers 207 square miles and serves 13 c...
2,MVSD,Mahoning Valley Sanitary District (MVSD),reservoir,Meander Creek Reservoir,25.22,2024.0,reported,available,SRC_MVSD_ACFR_2024,60.0,...,available,SRC_MVSD_ACFR_2024,220000.0,people,approximate,NaN,SRC_MVSD_ABOUT,regional_wholesale_water_system,core_fields_complete,MVSD is an analytical regional water-system ca...
3,ERIE_PERKINS,Erie County Water Division — Perkins District ...,purchased_great_lake,Purchased treated water from City of Sandusky ...,NaN,NaN,NaN,not_publicly_reported_current_perkins_specific,SRC_ERIE_PERKINS_CCR_2024,NaN,...,not_applicable,SRC_ERIE_MARCH146,17241.0,people,exact,NaN,SRC_ERIE_PWS_RECORD,distribution_pws,missing_current_average_daily_use,Perkins purchases finished water and does not ...
4,LAKE_BACON,Lake County Department of Utilities — East Sub...,great_lake,Lake Erie,3.00,NaN,reported,available_approximate,SRC_LAKE_BACON,9.0,...,available,SRC_LAKE_WATER_DIV,36000.0,people,greater_than,NaN,SRC_LAKE_BACON,treatment_plant_service_area,core_fields_complete,Average daily use is reported as roughly 3 MGD...


In [3]:
water_systems.shape

(7, 22)

In [4]:
water_systems.columns.tolist()

['water_system_id',
 'water_system',
 'source_type',
 'source_name',
 'average_daily_water_use_mgd',
 'water_use_year',
 'water_use_type',
 'water_use_status',
 'water_use_source_id',
 'treatment_capacity_mgd',
 'capacity_year',
 'capacity_type',
 'capacity_status',
 'capacity_source_id',
 'served_value',
 'served_unit',
 'served_qualifier',
 'served_year',
 'served_source_id',
 'system_scope',
 'research_status',
 'notes']

In [5]:
assert len(water_systems) == 7, "Expected exactly 7 frozen water-system cases."
assert water_systems["water_system_id"].is_unique, "water_system_id must be unique."

print("Water-system input validation passed.")

Water-system input validation passed.


In [6]:
capacities_mw = [50, 150, 300, 750]

water_intensity_scenarios = {
    "Very low water": 0.05,
    "Low water": 0.20,
    "Moderate water": 0.40,
    "Water-intensive cooling": 1.50
}

load_factor = 1.0

In [7]:
scenario_rows = []

for capacity in capacities_mw:
    for scenario_label, intensity in water_intensity_scenarios.items():
        scenario_rows.append({
            "it_capacity_mw": capacity,
            "load_factor": load_factor,
            "water_intensity_scenario": scenario_label,
            "water_intensity_l_kwh": intensity
        })

scenario_base = pd.DataFrame(scenario_rows)

scenario_base

,it_capacity_mw,load_factor,water_intensity_scenario,water_intensity_l_kwh
0,50,1.0,Very low water,0.05
1,50,1.0,Low water,0.20
2,50,1.0,Moderate water,0.40
3,50,1.0,Water-intensive cooling,1.50
4,150,1.0,Very low water,0.05
5,150,1.0,Low water,0.20
6,150,1.0,Moderate water,0.40
7,150,1.0,Water-intensive cooling,1.50
8,300,1.0,Very low water,0.05
9,300,1.0,Low water,0.20


In [8]:
assert len(scenario_base) == 16, "Expected exactly 16 base scenarios."

scenario_base

,it_capacity_mw,load_factor,water_intensity_scenario,water_intensity_l_kwh
0,50,1.0,Very low water,0.05
1,50,1.0,Low water,0.20
2,50,1.0,Moderate water,0.40
3,50,1.0,Water-intensive cooling,1.50
4,150,1.0,Very low water,0.05
5,150,1.0,Low water,0.20
6,150,1.0,Moderate water,0.40
7,150,1.0,Water-intensive cooling,1.50
8,300,1.0,Very low water,0.05
9,300,1.0,Low water,0.20


In [9]:
scenario_base["modeled_water_mgd"] = (
    scenario_base["it_capacity_mw"]
    * scenario_base["load_factor"]
    * scenario_base["water_intensity_l_kwh"]
    * 0.00634013
)

scenario_base["modeled_water_mg_year"] = (
    scenario_base["modeled_water_mgd"] * 365
)

scenario_base["modeled_water_bg_year"] = (
    scenario_base["modeled_water_mgd"] * 0.365
)

scenario_base

,it_capacity_mw,load_factor,water_intensity_scenario,water_intensity_l_kwh,modeled_water_mgd,modeled_water_mg_year,modeled_water_bg_year
0,50,1.0,Very low water,0.05,0.015850,5.785369,0.005785
1,50,1.0,Low water,0.20,0.063401,23.141475,0.023141
2,50,1.0,Moderate water,0.40,0.126803,46.282949,0.046283
3,50,1.0,Water-intensive cooling,1.50,0.475510,173.561059,0.173561
4,150,1.0,Very low water,0.05,0.047551,17.356106,0.017356
5,150,1.0,Low water,0.20,0.190204,69.424424,0.069424
6,150,1.0,Moderate water,0.40,0.380408,138.848847,0.138849
7,150,1.0,Water-intensive cooling,1.50,1.426529,520.683176,0.520683
8,300,1.0,Very low water,0.05,0.095102,34.712212,0.034712
9,300,1.0,Low water,0.20,0.380408,138.848847,0.138849


### 4B. Calculate modeled water demand

For each standardized scenario, modeled average water demand is calculated from IT capacity, load factor, and water intensity.

The model uses a fixed load factor of 1.0 and does not apply a PUE adjustment. Results represent hypothetical modeled water demand, not estimated water use by any individual Northeast Ohio facility.

In [10]:
scenario_base["modeled_water_mgd"] = (
    scenario_base["it_capacity_mw"]
    * scenario_base["load_factor"]
    * scenario_base["water_intensity_l_kwh"]
    * 0.00634013
)

scenario_base["modeled_water_mg_year"] = (
    scenario_base["modeled_water_mgd"] * 365
)

scenario_base["modeled_water_bg_year"] = (
    scenario_base["modeled_water_mgd"] * 0.365
)

scenario_base

,it_capacity_mw,load_factor,water_intensity_scenario,water_intensity_l_kwh,modeled_water_mgd,modeled_water_mg_year,modeled_water_bg_year
0,50,1.0,Very low water,0.05,0.015850,5.785369,0.005785
1,50,1.0,Low water,0.20,0.063401,23.141475,0.023141
2,50,1.0,Moderate water,0.40,0.126803,46.282949,0.046283
3,50,1.0,Water-intensive cooling,1.50,0.475510,173.561059,0.173561
4,150,1.0,Very low water,0.05,0.047551,17.356106,0.017356
5,150,1.0,Low water,0.20,0.190204,69.424424,0.069424
6,150,1.0,Moderate water,0.40,0.380408,138.848847,0.138849
7,150,1.0,Water-intensive cooling,1.50,1.426529,520.683176,0.520683
8,300,1.0,Very low water,0.05,0.095102,34.712212,0.034712
9,300,1.0,Low water,0.20,0.380408,138.848847,0.138849


## 5. Compare standardized scenarios across water systems

Each of the 16 standardized data-center scenarios is compared with each of the seven analytical water-system cases.

These comparisons are intended to show the relative scale of an equivalent hypothetical water demand in different local water-system contexts. They do not estimate whether a system has sufficient available capacity to accommodate a project.

Two comparisons are calculated where the appropriate denominator is available:

1. modeled water demand as a percentage of existing average daily water use; and
2. modeled water demand as a percentage of treatment capacity.

In [11]:
scenario_system_results = scenario_base.merge(
    water_systems,
    how="cross"
)

scenario_system_results.shape

(112, 29)

In [12]:
assert len(scenario_system_results) == 112, \
    "Expected 16 scenarios × 7 water systems = 112 rows."

assert not scenario_system_results.duplicated(
    subset=[
        "it_capacity_mw",
        "water_intensity_l_kwh",
        "water_system_id"
    ]
).any(), "Duplicate scenario-system combinations found."

print("Scenario-system cross join validation passed.")

Scenario-system cross join validation passed.


In [13]:
scenario_system_results[
    [
        "water_system_id",
        "water_system",
        "it_capacity_mw",
        "water_intensity_scenario",
        "water_intensity_l_kwh",
        "modeled_water_mgd",
        "average_daily_water_use_mgd",
        "treatment_capacity_mgd"
    ]
].head(14)

,water_system_id,water_system,it_capacity_mw,water_intensity_scenario,water_intensity_l_kwh,modeled_water_mgd,average_daily_water_use_mgd,treatment_capacity_mgd
0,CLE_WATER,Cleveland Division of Water / Cleveland Water,50,Very low water,0.05,0.015850,236.00,540.0
1,AKR_WATER,City of Akron Water Supply,50,Very low water,0.05,0.015850,30.44,67.0
2,MVSD,Mahoning Valley Sanitary District (MVSD),50,Very low water,0.05,0.015850,25.22,60.0
3,ERIE_PERKINS,Erie County Water Division — Perkins District ...,50,Very low water,0.05,0.015850,NaN,NaN
4,LAKE_BACON,Lake County Department of Utilities — East Sub...,50,Very low water,0.05,0.015850,3.00,9.0
5,PORT_SHALERSVILLE,Portage County Water Resources — Shalersville ...,50,Very low water,0.05,0.015850,NaN,4.0
6,CANTON_WATER,City of Canton Water Department,50,Very low water,0.05,0.015850,17.59,44.0
7,CLE_WATER,Cleveland Division of Water / Cleveland Water,50,Low water,0.20,0.063401,236.00,540.0
8,AKR_WATER,City of Akron Water Supply,50,Low water,0.20,0.063401,30.44,67.0
9,MVSD,Mahoning Valley Sanitary District (MVSD),50,Low water,0.20,0.063401,25.22,60.0


### 5C. Calculate water-system scale comparisons

For each scenario-system combination, modeled water demand is compared with:

- the water system's existing average daily water use, where available; and
- the water system's treatment capacity, where available.

Missing denominators are preserved rather than estimated or replaced with broader system values. These percentages are scale comparisons only and should not be interpreted as measures of available capacity or system headroom.

In [14]:
scenario_system_results["modeled_pct_existing_use"] = (
    scenario_system_results["modeled_water_mgd"]
    / scenario_system_results["average_daily_water_use_mgd"]
    * 100
)

scenario_system_results["modeled_pct_treatment_capacity"] = (
    scenario_system_results["modeled_water_mgd"]
    / scenario_system_results["treatment_capacity_mgd"]
    * 100
)

In [15]:
comparison_check = (
    scenario_system_results.groupby("water_system_id")[
        [
            "modeled_pct_existing_use",
            "modeled_pct_treatment_capacity"
        ]
    ]
    .count()
)

comparison_check

,modeled_pct_existing_use,modeled_pct_treatment_capacity
water_system_id,,
AKR_WATER,16,16
CANTON_WATER,16,16
CLE_WATER,16,16
ERIE_PERKINS,0,0
LAKE_BACON,16,16
MVSD,16,16
PORT_SHALERSVILLE,0,16


In [16]:
perkins = scenario_system_results[
    scenario_system_results["water_system_id"] == "ERIE_PERKINS"
]

shalersville = scenario_system_results[
    scenario_system_results["water_system_id"] == "PORT_SHALERSVILLE"
]

assert perkins["modeled_pct_existing_use"].isna().all()
assert perkins["modeled_pct_treatment_capacity"].isna().all()

assert shalersville["modeled_pct_existing_use"].isna().all()
assert shalersville["modeled_pct_treatment_capacity"].notna().all()

print("Missing-denominator validation passed.")

Missing-denominator validation passed.


In [17]:
check_row = scenario_system_results[
    (scenario_system_results["water_system_id"] == "LAKE_BACON")
    & (scenario_system_results["it_capacity_mw"] == 300)
    & (scenario_system_results["water_intensity_l_kwh"] == 0.40)
]

check_row[
    [
        "water_system",
        "modeled_water_mgd",
        "average_daily_water_use_mgd",
        "modeled_pct_existing_use",
        "treatment_capacity_mgd",
        "modeled_pct_treatment_capacity"
    ]
]

,water_system,modeled_water_mgd,average_daily_water_use_mgd,modeled_pct_existing_use,treatment_capacity_mgd,modeled_pct_treatment_capacity
74,Lake County Department of Utilities — East Sub...,0.760816,3.0,25.36052,9.0,8.453507


## 6. Validate model outputs

The following checks confirm that the standardized scenario model follows the frozen analytical methodology.

Validation covers:

- the expected scenario dimensions and row counts;
- the standardized capacity, water-intensity, and load-factor assumptions;
- uniqueness of scenario-system combinations;
- preservation of unavailable water-system denominators; and
- consistency of modeled water-demand calculations.

These checks are intended to catch accidental changes to the model before results are interpreted or exported.

In [18]:
# Expected frozen scenario assumptions
expected_capacities = {50, 150, 300, 750}
expected_intensities = {0.05, 0.20, 0.40, 1.50}

assert set(scenario_base["it_capacity_mw"]) == expected_capacities
assert set(scenario_base["water_intensity_l_kwh"]) == expected_intensities
assert (scenario_base["load_factor"] == 1.0).all()

assert len(scenario_base) == 16

print("Scenario assumption validation passed.")

Scenario assumption validation passed.


In [19]:
assert len(water_systems) == 7
assert water_systems["water_system_id"].is_unique
assert water_systems["water_system_id"].notna().all()

print("Water-system input validation passed.")

Water-system input validation passed.


In [20]:
assert len(scenario_system_results) == 112

assert not scenario_system_results.duplicated(
    subset=[
        "water_system_id",
        "it_capacity_mw",
        "water_intensity_l_kwh"
    ]
).any()

print("Scenario-system structure validation passed.")

Scenario-system structure validation passed.


In [21]:
perkins = scenario_system_results[
    scenario_system_results["water_system_id"] == "ERIE_PERKINS"
]

shalersville = scenario_system_results[
    scenario_system_results["water_system_id"] == "PORT_SHALERSVILLE"
]

assert perkins["modeled_pct_existing_use"].isna().all()
assert perkins["modeled_pct_treatment_capacity"].isna().all()

assert shalersville["modeled_pct_existing_use"].isna().all()
assert shalersville["modeled_pct_treatment_capacity"].notna().all()

print("Comparison eligibility validation passed.")

Comparison eligibility validation passed.


In [22]:
recalculated_mgd = (
    scenario_system_results["it_capacity_mw"]
    * scenario_system_results["load_factor"]
    * scenario_system_results["water_intensity_l_kwh"]
    * 0.00634013
)

assert np.allclose(
    scenario_system_results["modeled_water_mgd"],
    recalculated_mgd
)

print("Water-demand calculation validation passed.")

Water-demand calculation validation passed.


In [23]:
print("✓ Scenario assumptions: passed")
print("✓ Water-system inputs: passed")
print("✓ Scenario-system structure: passed")
print("✓ Comparison eligibility: passed")
print("✓ Water-demand calculations: passed")
print()
print("All model validation checks passed.")

✓ Scenario assumptions: passed
✓ Water-system inputs: passed
✓ Scenario-system structure: passed
✓ Comparison eligibility: passed
✓ Water-demand calculations: passed

All model validation checks passed.
